In [44]:
import dask.dataframe as dd
from dask_ml.model_selection import train_test_split
from dask_ml.preprocessing import StandardScaler
from dask_ml.wrappers import ParallelPostFit

import time

from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [45]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [46]:
df = dd.read_csv("iris.csv")

unique_labels = df['variety'].unique().compute().tolist()
mapping = {label: idx for idx, label in enumerate(sorted(unique_labels))}
df['label'] = df['variety'].map(mapping, meta=('label', 'int64'))


features = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = df[features]
y = df['label']

In [47]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, shuffle=True
)

In [48]:
pipelines = {
    "Decision Tree": Pipeline([
         ('scaler', StandardScaler()),
         ('clf', ParallelPostFit(DecisionTreeClassifier(max_depth=4)))
    ]),
    "Random Forest": Pipeline([
         ('scaler', StandardScaler()),
         ('clf', ParallelPostFit(RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)))
    ]),
    "Gradient Boosting": Pipeline([
         ('scaler', StandardScaler()),
         ('clf', ParallelPostFit(GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)))
    ])
}


In [51]:
results = {}

for name, pipeline in pipelines.items():
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    fit_time = time.time() - start_time
    
    # Les prédictions sont renvoyées comme une collection Dask
    y_pred = pipeline.predict(X_test)
    
    # Conversion en array numpy pour l'évaluation
    y_test_np = y_test.compute()
    y_pred_np = y_pred.compute()
    
    acc = accuracy_score(y_test_np, y_pred_np)
    cm = confusion_matrix(y_test_np, y_pred_np)
    
    results[name] = {"accuracy": acc, "fit_time": fit_time, "confusion_matrix": cm}

In [52]:
print("\n--- Model Comparison ---")
for name, res in results.items():
    print(f"\nModel: {name}")
    print(f"Accuracy: {res['accuracy']:.4f}")
    print(f"Fitting Time: {res['fit_time']:.4f} seconds")
    print("Confusion Matrix:")
    print(res["confusion_matrix"])


--- Model Comparison ---

Model: Decision Tree
Accuracy: 0.9375
Fitting Time: 0.1064 seconds
Confusion Matrix:
[[6 0 0]
 [0 6 0]
 [0 1 3]]

Model: Random Forest
Accuracy: 0.9375
Fitting Time: 0.1864 seconds
Confusion Matrix:
[[6 0 0]
 [0 6 0]
 [0 1 3]]

Model: Gradient Boosting
Accuracy: 0.9375
Fitting Time: 0.3026 seconds
Confusion Matrix:
[[6 0 0]
 [0 6 0]
 [0 1 3]]
